In [1]:
import import_ipynb
import data_analysis_classification

c:\Users\Natti\anaconda3\envs\ml_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ขนาดชุดข้อมูล: Train (8101, 27), Test (2026, 27)
ฟีเจอร์ที่ผ่านเกณฑ์โหวต (>=3): ['Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Total_Revolving_Bal', 'Contacts_Count_12_mon', 'Total_Trans_Amt', 'Months_Inactive_12_mon', 'Total_Relationship_Count']


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

df = pd.DataFrame(data_analysis_classification.data)
# Plain NumPy arrays, 1D labels (avoids Series/2D shape problems everywhere below)
X_train = np.asarray(data_analysis_classification.X_scaled, dtype=float)
X_test  = np.asarray(data_analysis_classification.X_test_scaled, dtype=float)
y_train = np.asarray(data_analysis_classification.y_train).ravel().astype(int)
y_test  = np.asarray(data_analysis_classification.y_test).ravel().astype(int)

print(X_train.shape, X_test.shape, "positive rate (train/test):", y_train.mean().round(3), y_test.mean().round(3))

(8101, 27) (2026, 27) positive rate (train/test): 0.161 0.16


In [3]:
# แปลงคอลัมน์ฟีเจอร์ทั้งหมด (ยกเว้นคอลัมน์สุดท้ายที่เป็น target) ให้เป็น numeric
feature_cols = df.columns[:-1]
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ดึงค่า X และ y พร้อมบังคับ dtype ให้ชัดเจน
X = df.iloc[:, :-1].to_numpy(dtype=np.float64)
y = df.iloc[:, -1].to_numpy()

# ตรวจสอบคลาสที่มีอยู่จริงทั้งหมดใน y
classes = np.unique(y)
print("Classes found in y:", classes)

# คำนวณ Mean Vector ของแต่ละคลาสตามค่าที่มีอยู่จริง
mean_vectors = []
for cl in classes:
    mean_vec = np.mean(X[y == cl], axis=0)
    mean_vectors.append(mean_vec)
    print('Mean Vector class %s: %s\n' % (cl, mean_vec))

Classes found in y: [0. 1.]
Mean Vector class 0.0: [1.55474894e-01 4.64162129e+01 5.20669490e-01 2.38596491e+00
 2.05263158e+00 1.33010688e+00 3.60076628e+01 3.81770518e+00
 2.34361766e+00 2.44141964e+00 8.34318248e+03 1.19697399e+03
 7.14620849e+03 7.65851381e-01 4.28166062e+03 6.29330510e+01
 7.15011494e-01 2.84253882e-01 1.41964106e-01 1.07481347e-01
 1.22000403e-01 9.40310546e-01 9.47771728e-03 1.81488203e-03
 4.83968542e-02 1.20790482e-01 8.79209518e-01]

Mean Vector class 1.0: [1.69000637e-01 4.63491407e+01 5.37873966e-01 2.29089752e+00
 2.04042011e+00 1.27021006e+00 3.60022279e+01 3.79789943e+00
 2.35041375e+00 2.48026735e+00 8.95394815e+03 1.12142521e+03
 7.83252295e+03 7.49772120e-01 4.55600255e+03 6.74936346e+01
 7.10776257e-01 2.62117759e-01 1.60089115e-01 1.12348822e-01
 0.00000000e+00 9.21705920e-01 1.49586251e-02 2.54614895e-03
 6.07893062e-02 0.00000000e+00 0.00000000e+00]



In [4]:

num_features = X.shape[1]
S_W = np.zeros((num_features, num_features))

for cl, mv in zip(classes, mean_vectors):
    class_sc_mat = np.zeros((num_features, num_features))
    for row in X[y == cl]:
        row = row.reshape(num_features, 1)
        mv_col = mv.reshape(num_features, 1)
        class_sc_mat += (row - mv_col).dot((row - mv_col).T)
    S_W += class_sc_mat

print('within-class Scatter Matrix:\n', S_W)

within-class Scatter Matrix:
 [[ 1.09238952e+03  4.11706120e+02  6.19527476e+01  1.21954471e+02
   3.49579718e+01 -5.59939421e+01  1.91908950e+02 -7.26135292e+02
   4.75001080e+02  6.77643496e+02 -6.60177159e+05 -6.46659729e+05
  -1.35174304e+04 -8.38754105e+01 -1.76053969e+06 -2.62825023e+04
  -2.06720055e+02 -1.54034273e+02  1.05383544e+01  1.04746567e+01
   1.39376891e+01  3.59472525e+00  2.74965006e+00  1.24872086e+00
  -7.59309617e+00 -6.12946158e+00  6.12946158e+00]
 [ 4.11706120e+02  5.16498929e+05  6.18290433e+02 -1.08017462e+04
  -2.65972445e+02  1.99523897e+03  4.04669740e+05 -1.24603917e+03
   3.30536926e+03 -2.19094342e+03  5.48897244e+05  1.14693724e+06
  -5.98039996e+05 -7.73357266e+02 -1.01290983e+07 -1.00841334e+05
  -5.66102782e+01  1.80015806e+02  2.41368327e+02  1.13911842e+02
  -5.04808832e+02  4.42087638e+02 -1.10971620e+02  1.44609581e+01
  -3.45576976e+02 -7.60311555e+02  7.60311555e+02]
 [ 6.19527476e+01  6.18290433e+02  2.01862438e+03  1.18217919e+01
   8.37952

In [5]:
overall_mean = np.mean(X, axis=0).reshape(num_features, 1)
S_B = np.zeros((num_features, num_features))

for cl, mean_vec in zip(classes, mean_vectors):
    n = X[y == cl].shape[0]
    mean_vec_col = mean_vec.reshape(num_features, 1)
    S_B += n * (mean_vec_col - overall_mean).dot((mean_vec_col - overall_mean).T)

print('between-class Scatter Matrix:\n', S_B)

between-class Scatter Matrix:
 [[ 3.51871329e-01 -1.74488088e+00  4.47573344e-01 -2.47317223e+00
  -3.17680469e-01 -1.55821194e+00 -1.41390007e-01 -5.15245467e-01
   1.76799702e-01  1.01062057e+00  1.58890303e+04 -1.96539670e+03
   1.78544270e+04 -4.18300970e-01  7.13698784e+03  1.18643293e+02
  -1.10179424e-01 -5.75869818e-01  4.71520971e-01  1.26627062e-01
  -3.17383274e+00 -4.83998180e-01  1.42585468e-01  1.90238624e-02
   3.22388850e-01 -3.14235671e+00 -2.28725797e+01]
 [-1.74488088e+00  8.65262108e+00 -2.21945383e+00  1.22641164e+01
   1.57533317e+00  7.72695584e+00  7.01133337e-01  2.55503046e+00
  -8.76725083e-01 -5.01152653e+00 -7.87914866e+04  9.74612833e+03
  -8.85376149e+04  2.07429621e+00 -3.53913280e+04 -5.88335552e+02
   5.46364406e-01  2.85565817e+00 -2.33820678e+00 -6.27925952e-01
   1.57385942e+01  2.40007952e+00 -7.07061463e-01 -9.43366820e-02
  -1.59868137e+00  1.55825090e+01  1.13421935e+02]
 [ 4.47573344e-01 -2.21945383e+00  5.69304408e-01 -3.14582598e+00
  -4.0408

In [6]:
# Solving the generalized eigenvalue problem for the matrix S_W^{-1} S_B
# Use the actual feature dimension instead of a hard-coded 2x2 assumption.
A = np.linalg.pinv(S_W).dot(S_B)
eig_vals, eig_vecs = np.linalg.eig(A)

for i in range(len(eig_vals)):
    eigvec_sc = eig_vecs[:, i].reshape(num_features, 1)
    print('\nEigenvector {}: \n{}'.format(i + 1, eigvec_sc.real))
    print('Eigenvalue {:}: {:.2e}'.format(i + 1, eig_vals[i].real))


Eigenvector 1: 
[[-9.99713378e-01]
 [ 3.47691605e-05]
 [-2.95512665e-04]
 [-4.22491391e-04]
 [-1.67872700e-05]
 [-2.89175071e-04]
 [ 4.36624500e-06]
 [ 5.09191171e-05]
 [-9.28849270e-05]
 [ 2.32727815e-04]
 [-1.35239700e-07]
 [-3.18012744e-07]
 [ 1.82847863e-07]
 [-1.39771033e-03]
 [-3.89650393e-07]
 [ 9.58725641e-05]
 [ 9.20173477e-04]
 [ 5.36788675e-04]
 [ 9.60439554e-04]
 [-2.29906795e-04]
 [-6.53790446e-03]
 [-1.27026861e-03]
 [ 2.17312249e-03]
 [-2.73057689e-04]
 [-6.29796191e-04]
 [ 1.61108953e-02]
 [-1.61108953e-02]]
Eigenvalue 1: 0.00e+00

Eigenvector 2: 
[[-9.74624108e-02]
 [-1.44538236e-03]
 [ 1.22847025e-02]
 [ 1.75633117e-02]
 [ 6.97860507e-04]
 [ 1.20212435e-02]
 [-1.81508366e-04]
 [-2.11674923e-03]
 [ 3.86130217e-03]
 [-9.67468506e-03]
 [ 5.62202465e-06]
 [ 1.32200491e-05]
 [-7.60113480e-06]
 [ 5.81039582e-02]
 [ 1.61980846e-05]
 [-3.98550068e-03]
 [-3.82523619e-02]
 [-2.23147430e-02]
 [-3.99262556e-02]
 [ 9.55741298e-03]
 [ 2.71786020e-01]
 [ 5.28061020e-02]
 [-9.033847

In [7]:
# Make a list of (eigenvalue, eigenvector) tuples
eig_pairs = [(np.abs(eig_vals[i]), eig_vecs[:,i]) for i in range(len(eig_vals))]

# Sort the (eigenvalue, eigenvector) tuples from high to low
eig_pairs = sorted(eig_pairs, key=lambda k: k[0], reverse=True)

# Visually confirm that the list is correctly sorted by decreasing eigenvalues

print('Eigenvalues in decreasing order:\n')
for i in eig_pairs:
    print(i[0])

Eigenvalues in decreasing order:

0.5627258858680986
8.052450369275478e-17
1.4948540465792932e-17
1.2918344358863791e-17
1.2918344358863791e-17
1.0369707469498594e-17
8.845034281338943e-18
3.3442926839373832e-18
3.2067293668988328e-18
2.2294379735160167e-18
1.8543721079862827e-18
1.8543721079862827e-18
1.2864410536946474e-18
1.2864410536946474e-18
1.1659693205496388e-18
1.088623554998058e-18
4.4480303666695535e-19
3.2028825333662245e-19
3.2028825333662245e-19
1.8042944525049962e-19
1.346284266172944e-19
1.346284266172944e-19
4.2254319776106607e-20
1.2646500207282104e-20
1.2646500207282104e-20
1.2592492827568254e-20
0.0


In [8]:
print('Variance explained:\n')
eigv_sum = np.sum(np.abs(eig_vals))
if np.isclose(eigv_sum, 0):
    raise ValueError('LDA failed: total eigenvalue sum is zero; the within-class scatter is singular.')

for i, j in enumerate(eig_pairs):    print('eigenvalue {0:}: {1:.2%}'.format(i + 1, (j[0] / eigv_sum).real))

Variance explained:

eigenvalue 1: 100.00%
eigenvalue 2: 0.00%
eigenvalue 3: 0.00%
eigenvalue 4: 0.00%
eigenvalue 5: 0.00%
eigenvalue 6: 0.00%
eigenvalue 7: 0.00%
eigenvalue 8: 0.00%
eigenvalue 9: 0.00%
eigenvalue 10: 0.00%
eigenvalue 11: 0.00%
eigenvalue 12: 0.00%
eigenvalue 13: 0.00%
eigenvalue 14: 0.00%
eigenvalue 15: 0.00%
eigenvalue 16: 0.00%
eigenvalue 17: 0.00%
eigenvalue 18: 0.00%
eigenvalue 19: 0.00%
eigenvalue 20: 0.00%
eigenvalue 21: 0.00%
eigenvalue 22: 0.00%
eigenvalue 23: 0.00%
eigenvalue 24: 0.00%
eigenvalue 25: 0.00%
eigenvalue 26: 0.00%
eigenvalue 27: 0.00%


In [9]:
# Choosing k eigenvectors with the largest eigenvalues
# Make this work for the actual number of features instead of assuming exactly 2 features.
W = np.hstack([eig_pairs[0][1].reshape(num_features, 1)])
print('Matrix W:\n', W.real)

Matrix W:
 [[-9.74624108e-02]
 [-1.44538236e-03]
 [ 1.22847025e-02]
 [ 1.75633117e-02]
 [ 6.97860507e-04]
 [ 1.20212435e-02]
 [-1.81508366e-04]
 [-2.11674923e-03]
 [ 3.86130217e-03]
 [-9.67468506e-03]
 [ 5.62202465e-06]
 [ 1.32200491e-05]
 [-7.60113480e-06]
 [ 5.81039582e-02]
 [ 1.61980846e-05]
 [-3.98550068e-03]
 [-3.82523619e-02]
 [-2.23147430e-02]
 [-3.99262556e-02]
 [ 9.55741298e-03]
 [ 2.71786020e-01]
 [ 5.28061020e-02]
 [-9.03384741e-02]
 [ 1.13512309e-02]
 [ 2.61811413e-02]
 [-6.69743056e-01]
 [ 6.69743056e-01]]


In [10]:
#Transforming the samples onto the new subspace
X_lda = X.dot(W)

In [11]:
X_lda

array([[ 0.5294155 +0.j],
       [-0.23334716+0.j],
       [ 0.43965874+0.j],
       ...,
       [-0.30627744+0.j],
       [ 0.54328704+0.j],
       [-0.21168421+0.j]])

In [12]:
class CatBoostClassifier:
    def __init__(self, learning_rate=0.1, n_estimators=100, max_depth=3,
                 l2_leaf_reg=3.0, border_count=32, cat_features=None,
                 prior_weight=1.0, random_state=0):
        self.learning_rate = learning_rate
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.l2_leaf_reg = l2_leaf_reg
        self.border_count = border_count
        self.cat_features = cat_features
        self.prior_weight = prior_weight
        self.random_state = random_state

    # ---------------- helpers ----------------
    @staticmethod
    def _softmax(z):
        z = z - z.max(axis=1, keepdims=True)          # numerically stable
        e = np.exp(z)
        return e / e.sum(axis=1, keepdims=True)

    def _as_array(self, X):
        X = np.asarray(X)
        if self._cats:                                 # mixed types -> object copy
            X = np.array(X, dtype=object, copy=True)
        else:
            X = X.astype(float)
        return X

    def _ordered_ts(self, codes, target, perm, prior):
        """TS for each row from rows BEFORE it in `perm` only."""
        cp, tp = codes[perm], target[perm]
        idx = np.argsort(cp, kind="stable")
        sc, st = cp[idx], tp[idx]
        n = len(sc)
        starts = np.r_[0, np.flatnonzero(np.diff(sc)) + 1]
        grp_start = np.repeat(starts, np.diff(np.r_[starts, n]))
        cs = np.cumsum(st)
        base = np.where(grp_start > 0, cs[grp_start - 1], 0.0)
        sum_before = cs - st - base
        cnt_before = np.arange(n) - grp_start
        ts_sorted = (sum_before + self.prior_weight * prior) / (cnt_before + self.prior_weight)
        tmp = np.empty(n)
        tmp[idx] = ts_sorted
        out = np.empty(n)
        out[perm] = tmp
        return out

    def _matrix(self, X, train):
        """Numeric matrix: numeric cols as-is, each cat col -> K TS columns."""
        n, d = X.shape
        cols = []
        for j in range(d):
            if j not in self._cats:
                cols.append(X[:, j].astype(float))
            elif train:
                cols.extend(self._train_ts[j])
            else:
                m = self._cat_stats[j]
                vals = X[:, j].astype(str)
                for k in range(self.K):
                    out = np.full(n, self._priors[k])
                    for i, v in enumerate(vals):
                        if v in m:
                            s, c = m[v]
                            out[i] = (s[k] + self.prior_weight * self._priors[k]) / (c + self.prior_weight)
                    cols.append(out)
        return np.column_stack(cols)

    def _quantize(self, M, fit=False):
        n, d = M.shape
        if fit:
            qs = np.linspace(0, 1, self.border_count + 1)[1:-1]
            self._borders = [np.unique(np.quantile(M[:, j], qs)) for j in range(d)]
        Xb = np.empty((n, d), dtype=np.int64)
        for j in range(d):
            Xb[:, j] = np.searchsorted(self._borders[j], M[:, j], side="left")
        return Xb

    # ---------------- one oblivious tree ----------------
    def _build_tree(self, Xb, G, H):
        n, d = Xb.shape
        K = self.K
        B = max(len(b) for b in self._borders) + 1
        leaf = np.zeros(n, dtype=np.int64)
        splits = []
        for level in range(self.max_depth):
            L = 2 ** level
            best_score, best = -np.inf, None
            for j in range(d):
                nb = len(self._borders[j]) + 1
                if nb < 2:
                    continue
                key = leaf * B + Xb[:, j]
                score = np.zeros(nb - 1)
                for k in range(K):
                    g = np.bincount(key, weights=G[:, k], minlength=L * B).reshape(L, B)
                    h = np.bincount(key, weights=H[:, k], minlength=L * B).reshape(L, B)
                    gl = np.cumsum(g, 1)[:, :nb - 1]
                    hl = np.cumsum(h, 1)[:, :nb - 1]
                    gr = g.sum(1, keepdims=True) - gl
                    hr = h.sum(1, keepdims=True) - hl
                    score += (gl**2 / (hl + self.l2_leaf_reg)
                              + gr**2 / (hr + self.l2_leaf_reg)).sum(0)
                t = int(np.argmax(score))
                if score[t] > best_score:
                    best_score, best = score[t], (j, t)
            if best is None:
                break
            splits.append(best)
            leaf = leaf * 2 + (Xb[:, best[0]] > best[1])
        L = 2 ** len(splits)
        values = np.zeros((L, K))
        for k in range(K):
            g = np.bincount(leaf, weights=G[:, k], minlength=L)
            h = np.bincount(leaf, weights=H[:, k], minlength=L)
            values[:, k] = g / (h + self.l2_leaf_reg)      # Newton step per leaf
        return splits, values, leaf

    @staticmethod
    def _leaf_index(Xb, splits):
        idx = np.zeros(len(Xb), dtype=np.int64)
        for j, t in splits:
            idx = idx * 2 + (Xb[:, j] > t)
        return idx

    # ---------------- public API ----------------
    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        self._cats = set(self.cat_features or [])
        X = self._as_array(X)
        self.classes_, yi = np.unique(np.asarray(y), return_inverse=True)
        self.K = len(self.classes_)
        n = len(yi)
        one_hot_y = np.eye(self.K)[yi]
        self._priors = one_hot_y.mean(axis=0)

        # ordered target statistics (single random permutation)
        perm = rng.permutation(n)
        self._train_ts, self._cat_stats = {}, {}
        for j in self._cats:
            uniq, codes = np.unique(X[:, j].astype(str), return_inverse=True)
            self._train_ts[j] = [self._ordered_ts(codes, one_hot_y[:, k], perm, self._priors[k])
                                 for k in range(self.K)]
            self._cat_stats[j] = {u: (one_hot_y[codes == c].sum(0), (codes == c).sum())
                                  for c, u in enumerate(uniq)}

        Xb = self._quantize(self._matrix(X, train=True), fit=True)

        self._base = np.log(np.clip(self._priors, 1e-9, None))
        predictions = np.tile(self._base, (n, 1))
        self.trees = []
        for _ in range(self.n_estimators):
            P = self._softmax(predictions)
            G = one_hot_y - P                  # residuals (negative gradient)
            H = P * (1 - P)                    # diagonal hessian
            splits, values, leaf = self._build_tree(Xb, G, H)
            predictions += self.learning_rate * values[leaf]
            self.trees.append((splits, values))
        return self

    def _raw(self, samples):
        X = self._as_array(samples)
        Xb = self._quantize(self._matrix(X, train=False))
        out = np.tile(self._base, (len(Xb), 1))
        for splits, values in self.trees:
            out += self.learning_rate * values[self._leaf_index(Xb, splits)]
        return out

    def predict_proba(self, samples):
        return self._softmax(self._raw(samples))

    def predict(self, samples):
        return self.classes_[np.argmax(self._raw(samples), axis=1)]



In [13]:
class AdaBoost:
    def __init__(self, regression=False, n_estimators=50, learning_rate=1.0, random_state=0):
        self.regression = regression
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.random_state = random_state
        self.stumps = []
        self.weights = []
        self.classes_ = None
        self._label_to_index = {}

    def _update_weights(self, args):
        if self.regression:
            weights, beta, loss_function = args
            weights *= beta ** (1 - loss_function)
        else:
            weights, wrong_predictions, stump_weight = args
            weights[wrong_predictions] *= np.exp(stump_weight)

        return weights

    @staticmethod
    def _normalize(weights: np.ndarray):
        return weights / sum(weights)

    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self._label_to_index = {label: idx for idx, label in enumerate(self.classes_)}
        n_samples = len(y)
        self.K = len(self.classes_)
        weights = np.ones(n_samples) / n_samples

        for _ in range(self.n_estimators):
            if self.regression:
                stump = DecisionTreeRegressor(max_depth=3, random_state=self.random_state)
                stump_predictions = stump.fit(X, y, sample_weight=weights).predict(X)

                abs_instance_errors = abs(stump_predictions - y)
                adjusted_instance_errors = abs_instance_errors / max(abs_instance_errors)
                adjusted_stump_error = sum(weights * adjusted_instance_errors)

                if adjusted_stump_error >= 0.5:
                    self.stumps.pop(-1)
                    break

                beta = adjusted_stump_error / (1 - adjusted_stump_error)
                stump_weight = self.learning_rate * np.log(1 / beta)
                args = [weights, beta, adjusted_instance_errors]
            else:
                stump = DecisionTreeClassifier(max_depth=1, random_state=self.random_state)
                stump_predictions = stump.fit(X, y, sample_weight=weights).predict(X)

                wrong_predictions = stump_predictions != y
                r = sum(weights[wrong_predictions])
                stump_weight = self.learning_rate * np.log((1 - r) / r) + np.log(self.K - 1)
                args = [weights, wrong_predictions, stump_weight]

            self.stumps.append(stump)
            self.weights.append(stump_weight)
            weights = self._update_weights(args)
            weights = self._normalize(weights)

    def _max_weighted_votes(self, samples):
        n_samples = len(samples)
        sample_indexes = np.array(range(n_samples))
        prediction_weights = np.zeros((n_samples, self.K))

        for i in range(self.n_estimators):
            stump_prediction = np.asarray(self.stumps[i].predict(samples))
            mapped_predictions = np.array([self._label_to_index[pred] for pred in stump_prediction], dtype=int)
            prediction_weights[sample_indexes, mapped_predictions] += self.weights[i]

        best_index = np.argmax(prediction_weights, axis=1)
        return self.classes_[best_index]

    def _weighted_median_prediction(self, samples):
        n_samples = len(samples)
        sample_indexes = np.arange(n_samples)
        predictions = np.array([stump.predict(samples) for stump in self.stumps]).T
        sorted_pred_indexes = np.argsort(predictions, axis=1)

        cumsum_weights = np.array(self.weights)[sorted_pred_indexes].cumsum(axis=1)
        is_over_median = cumsum_weights >= 0.5 * sum(self.weights)
        median_indexes = is_over_median.argmax(axis=1)
        median_prediction_indexes = sorted_pred_indexes[sample_indexes, median_indexes]

        return predictions[sample_indexes, median_prediction_indexes]

    def predict_proba(self, samples):
        n_samples = len(samples)
        scores = np.zeros((n_samples, self.K))

        for stump, weight in zip(self.stumps, self.weights):
            predictions = np.asarray(stump.predict(samples))
            mapped_predictions = np.array([self._label_to_index[pred] for pred in predictions], dtype=int)

            for i, pred_idx in enumerate(mapped_predictions):
                scores[i, pred_idx] += weight

        scores = scores / scores.sum(axis=1, keepdims=True)
        return scores

    def predict(self, samples):
        if self.regression:
            return self._weighted_median_prediction(samples)
        else:
            return self._max_weighted_votes(samples)

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

# 1. ปรับค่าข้อมูล X_lda ให้เป็น real number (ตัด imaginary part ออกหากมี)
X_lda_real = X_lda.real

# 2. แบ่งข้อมูล Train และ Test
X_tr, X_te, y_tr, y_te = train_test_split(X_lda_real, y, test_size=0.3, random_state=42, stratify=y)

# 3. เทรนโมเดล Classifier
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_tr, y_tr)

cb_clf = CatBoostClassifier(
    n_estimators=100,
    max_depth=3
).fit(X_tr, y_tr)

ada_clf = AdaBoost(
    regression=False,
    n_estimators=50,
    learning_rate=1.0,
    random_state=0
)
ada_clf.fit(X_tr, y_tr)
# 4. ทำนายผล
y_pred = clf.predict(X_te)
cb_y_pred = cb_clf.predict(X_te)
ada_y_pred = ada_clf.predict(X_te)
y_proba = clf.predict_proba(X_te)[:, 1] if len(np.unique(y)) == 2 else None
cb_y_proba = cb_clf.predict_proba(X_te)[:, 1] if len(np.unique(y)) == 2 else None
ada_y_proba = ada_clf.predict_proba(X_te)[:, 1] if len(np.unique(y)) == 2 else None
# 5. คำนวณ Evaluation Metrics
print("Accuracy :", accuracy_score(y_te, y_pred))
print("Precision:", precision_score(y_te, y_pred, average='weighted'))
print("Recall   :", recall_score(y_te, y_pred, average='weighted'))
print("F1-score :", f1_score(y_te, y_pred, average='weighted'))
if y_proba is not None:
    # หากคลาสเป็น 1 และ 2 ให้แปลงเป็น 0 และ 1 สำหรับ roc_auc_score
    y_te_binary = np.where(y_te == 2, 1, 0)
    print("ROC-AUC  :", roc_auc_score(y_te_binary, y_proba))
print("\nFull Report:\n", classification_report(y_te, y_pred))
print("\nCatBoost Classifier Metrics:")
print("Accuracy :", accuracy_score(y_te, cb_y_pred))
print("Precision:", precision_score(y_te, cb_y_pred, average='weighted'))
print("Recall   :", recall_score(y_te, cb_y_pred, average='weighted'))
print("F1-score :", f1_score(y_te, cb_y_pred, average='weighted'))


if cb_y_proba is not None:
    # หากคลาสเป็น 1 และ 2 ให้แปลงเป็น 0 และ 1 สำหรับ roc_auc_score
    y_te_binary = np.where(y_te == 2, 1, 0)
    print("ROC-AUC  :", roc_auc_score(y_te_binary, cb_y_proba))
print("\nFull Report:\n", classification_report(y_te, cb_y_pred))

print("\nAdaBoost Classifier Metrics:")
print("Accuracy :", accuracy_score(y_te, ada_y_pred))
print("Precision:", precision_score(y_te, ada_y_pred, average='weighted'))
print("Recall   :", recall_score(y_te, ada_y_pred, average='weighted'))
print("F1-score :", f1_score(y_te, ada_y_pred, average='weighted'))
if ada_y_proba is not None:
    # หากคลาสเป็น 1 และ 2 ให้แปลงเป็น 0 และ 1 สำหรับ roc_auc_score
    y_te_binary = np.where(y_te == 2, 1, 0)
    print("ROC-AUC  :", roc_auc_score(y_te_binary, ada_y_proba))    
print("\nFull Report:\n", classification_report(y_te, ada_y_pred))

Accuracy : 0.9995886466474702
Precision: 0.9995889229089562
Recall   : 0.9995886466474702
F1-score : 0.9995886066237463
ROC-AUC  : nan

Full Report:
               precision    recall  f1-score   support

         0.0       1.00      1.00      1.00      1488
         1.0       1.00      1.00      1.00       943

    accuracy                           1.00      2431
   macro avg       1.00      1.00      1.00      2431
weighted avg       1.00      1.00      1.00      2431


CatBoost Classifier Metrics:
Accuracy : 0.9786096256684492
Precision: 0.9797275145782387
Recall   : 0.9786096256684492
F1-score : 0.9787063949477708
ROC-AUC  : nan

Full Report:
               precision    recall  f1-score   support

         0.0       1.00      0.97      0.98      1488
         1.0       0.95      1.00      0.97       943

    accuracy                           0.98      2431
   macro avg       0.97      0.98      0.98      2431
weighted avg       0.98      0.98      0.98      2431


AdaBoost Classi

c:\Users\Natti\anaconda3\envs\ml_env\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\Natti\anaconda3\envs\ml_env\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\Natti\anaconda3\envs\ml_env\lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
